# BiciTierra Market · Predicció mensual amb separació temporal

Compareu una mitjana de grup, el mateix mes de l’any anterior i regressió lineal regularitzada. El model treballa amb dades sintètiques; no és una previsió de mercat real. Requereix `pandas` i `numpy`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('..') / 'dades'
sales = pd.read_csv(DATA / 'sales_history.csv')
sales['month'] = pd.to_datetime(sales['month'], format='%Y-%m')
KEY = ['channel', 'product_line', 'region']
sales = sales.sort_values(KEY + ['month']).reset_index(drop=True)
assert not sales.duplicated(['month'] + KEY).any()
sales.head()

## 1. Retards i períodes

Ordenem cada sèrie i usem `shift`, mai el valor del mateix mes. El primer any queda fora de l’ajust perquè falta `lag_12`. Entrenament 2022–2024, validació 2025 i prova gener–juny 2026.

In [ ]:
# Els retards es calculen dins de cada sèrie i només miren mesos anteriors.
for lag in [1, 12]:
    sales[f'lag_{lag}'] = sales.groupby(KEY)['sales_units'].shift(lag)
usable = sales.dropna(subset=['lag_1', 'lag_12']).copy()
train = usable[usable['month'] < '2025-01-01'].copy()
valid = usable[(usable['month'] >= '2025-01-01') & (usable['month'] < '2026-01-01')].copy()
test = usable[usable['month'] >= '2026-01-01'].copy()
print({'train': len(train), 'valid': len(valid), 'test': len(test)})

## 2. Referències i mètriques

La mitjana s’aprén només amb entrenament. MAE és error mitjà en unitats; WAPE és error absolut total respecte al volum real total. Limitem totes les prediccions al rang físic 0–estoc amb la mateixa regla.

In [ ]:
def score(y, pred):
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    return {'MAE_units': float(np.mean(np.abs(y-pred))),
            'WAPE_pct': float(100*np.abs(y-pred).sum()/np.abs(y).sum())}

group_mean = train.groupby(KEY)['sales_units'].mean().rename('prediction')
def mean_prediction(frame):
    merged = frame.merge(group_mean.reset_index(), on=KEY, how='left', validate='many_to_one')
    return merged['prediction'].fillna(train['sales_units'].mean()).to_numpy()

def cap(frame, prediction):
    return np.clip(prediction, 0, frame['stock_units'].to_numpy())

### Comprovació de setmana 2

Calculeu les referències amb validació; reserveu la regressió per a setmana 3 i el test per a setmana 5.


In [ ]:
reference_results = pd.DataFrame([
    {'model': 'group_mean', **score(valid['sales_units'], cap(valid, mean_prediction(valid)))},
    {'model': 'seasonal_lag12', **score(valid['sales_units'], cap(valid, valid['lag_12']))},
])
reference_results.round(2)


> Setmana 3. Continueu ací després de revisar CP2.

## 3. Model lineal

La regularització penalitza coeficients massa grans. Es prova el paràmetre `alpha` en validació. Els predictors exclosos inclouen sessions, ingressos, despesa real, descompte real, estoc final i ruptura del mateix mes. Les variables retardades sí que eren conegudes.

En treball posterior podeu reproduir este procés amb un `Pipeline` de scikit-learn i comparar-lo amb un arbre, mantenint exactament les mateixes dates.

In [ ]:
# Regressió lineal regularitzada (ridge) amb NumPy: no necessita scikit-learn.
# Imputació, codificació i escala apreses només amb entrenament.
NUM = ['marketing_budget','planned_discount','stock_units','list_price',
       'days_in_month','lag_1','lag_12']
CAT = KEY + ['campaign']
def design(frame):
    x = frame[NUM].copy()
    month = frame['month'].dt.month
    x['month_sin'] = np.sin(2*np.pi*month/12)
    x['month_cos'] = np.cos(2*np.pi*month/12)
    x['time_index'] = (frame['month'].dt.year-2021)*12+month-1
    for product in ['urban','mountain','electric']:
        x[f'{product}_sin'] = x['month_sin']*(frame['product_line']==product)
        x[f'{product}_cos'] = x['month_cos']*(frame['product_line']==product)
    return pd.concat([x, pd.get_dummies(frame[CAT], dtype=float)], axis=1)

def fit_ridge(frame, alpha):
    x = design(frame)
    columns = x.columns
    medians = x.median()
    x = x.fillna(medians)
    center, scale = x.mean(), x.std().replace(0, 1)
    z = ((x-center)/scale).to_numpy(dtype=float)
    z = np.column_stack([np.ones(len(z)), z])
    penalty = np.eye(z.shape[1])*alpha
    penalty[0,0] = 0 # La constant no es penalitza.
    beta = np.linalg.solve(z.T@z+penalty, z.T@frame['sales_units'].to_numpy())
    return columns, medians, center, scale, beta

def predict_ridge(frame, model):
    columns, medians, center, scale, beta = model
    x = design(frame).reindex(columns=columns, fill_value=0).fillna(medians)
    z = ((x-center)/scale).to_numpy(dtype=float)
    return np.column_stack([np.ones(len(z)), z])@beta

## 4. Comparar en validació

Trieu el model amb 2025. No mireu el test per canviar variables o paràmetres.

In [ ]:
validation_results = []
validation_results.append({'model':'group_mean', **score(valid['sales_units'],cap(valid,mean_prediction(valid)))})
validation_results.append({'model':'seasonal_lag12', **score(valid['sales_units'],cap(valid,valid['lag_12']))})
models = {}
for alpha in [1.0, 10.0, 100.0]:
    name = f'ridge_{alpha:g}'
    models[name] = fit_ridge(train, alpha)
    validation_results.append({'model':name, **score(valid['sales_units'],cap(valid,predict_ridge(valid,models[name])))})
comparison = pd.DataFrame(validation_results).sort_values('MAE_units').reset_index(drop=True)
chosen = comparison.iloc[0]['model']
print(comparison.round(2).to_string(index=False))
print('Model seleccionat només amb validació:', chosen)

> Setmana 5. Atureu-vos ací fins que el model estiga triat i revisat en CP3. No useu el test per afinar decisions.

## 5. Prova final

Fixada la decisió anterior, avaluem 2026 una sola vegada. L’ajust queda congelat; els retards s’actualitzen amb mesos que ja s’han observat. És predicció mensual successiva, no una predicció simultània de sis mesos.

In [ ]:
def selected_prediction(frame):
    if chosen == 'group_mean': return cap(frame,mean_prediction(frame))
    if chosen == 'seasonal_lag12': return cap(frame,frame['lag_12'])
    return cap(frame,predict_ridge(frame,models[chosen]))

# Prova d'un mes vista: el retard 1 de juny usa maig, ja observat en aquell moment.
test_prediction = selected_prediction(test)
print('Prova final:', score(test['sales_units'], test_prediction))
errors = test[['month']+KEY+['sales_units']].copy()
errors['prediction'] = test_prediction
errors['abs_error'] = abs(errors['sales_units']-errors['prediction'])
errors.sort_values('abs_error',ascending=False).head(10)

## 6. Aplicar a juliol de 2026

No hi ha objectiu real per comparar. Usem el mateix model triat i congelat per fer el procés fàcil de seguir. Reentrenar amb tot l’històric abans de producció seria una ampliació posterior; caldria distingir-ne les mètriques de les de la prova.

In [ ]:
future = pd.read_csv(DATA / 'sales_forecast_input.csv')
future['month'] = pd.to_datetime(future['month'],format='%Y-%m')
# Correspondència exacta de mes i sèrie, sense usar files d'altres grups.
for lag in [1,12]:
    past = sales[['month']+KEY+['sales_units']].copy()
    past['month'] = past['month'] + pd.DateOffset(months=lag)
    past = past.rename(columns={'sales_units':f'lag_{lag}'})
    future = future.merge(past,on=['month']+KEY,how='left',validate='one_to_one')
assert future[['lag_1','lag_12']].notna().all().all()
forecast = future[['month']+KEY].copy()
forecast['predicted_sales_units'] = np.rint(selected_prediction(future)).astype(int)
forecast
# Per guardar, descomenta: forecast.to_csv('prediccions_juliol_2026.csv', index=False)

## Entrega

Expliqueu el període de cada partició, la diferència entre predictors planificats i observats, el model triat i el seu error final. Mostreu un cas on falle. No presenteu la previsió de juliol com a resultat validat: la resposta encara no està disponible en el cas.